In [2]:
# Imports.
import numpy as np;
import os;
import glob;
import pandas as pd;
import xarray as xr;
import matplotlib.pyplot as plt;
import h5_reader_xr as reader;
import phi2D_utilities as utils;

# Styling.
plt.style.use("ggplot");

In [3]:
# Test inputs - delete when done!
BASE_PATH = "/zhisongqu_data/seth/GYSELA/raw/fresh_batch_9.0";
PREFIX = "DN_NUSTARSCAN_RECTEST";
effective_radius = 0.7

In [ ]:
def parameter_scan_delta_f_evolution(base_directory, folder_prefix, dt_diag):

	# This method presumes we process parameter scans, in the same vein as `parameter_scan_analysis_phi2D`.
	# `folder_prefix` should be of the form "DN_*_*_[parameter value]" (DN is the standard GYSELA format, not necessarily invoked here).
	search_pattern = os.path.join(base_directory, f"{folder_prefix}_*");
	# Match search pattern, return list in ascending order.
	matching_directories = sorted(glob.glob(search_pattern));

	if not matching_directories:

		print(f"No directories found matching pattern: {search_pattern}");
		return;

	results = [];

	for directory in matching_directories:

		folder_basename = os.path.basename(directory);
		# Split the folder name, taking the last entry as that corresponding to the parameter value.
		parameter_value_string = folder_basename.split("_")[-1];
		parameter_value = float(parameter_value_string);
		print(f"Processing {folder_basename} with parameter value: {parameter_value}");

		f2D_list = reader.fetch_f2D_data(directory);
		delta_t = reader.fetch_delta_t(directory);

		vpar, delta_f_vpar_time = utils.compute_delta_f_vpar(f2D_list);
		stride = utils.calculate_stride(delta_t, dt_diag);
		time_range = np.arange(len(delta_f_vpar_time)) * stride;
		results.append({
			"folder_name": folder_basename,
			"time_range": time_range,
			"vpar": vpar,
			"delta_f": delta_f_vpar_time,
		});

	dataframe_results = pd.Dataframe(results);
	return dataframe_results;

In [ ]:
def plot_parameter_scan_delta_f_evolution(dataframe, n_columns = 2, parameter_label = "param"):

	n_runs = len(dataframe);
	n_columns = min(n_columns, n_runs);
	n_rows = int(np.ceil(n_runs / n_columns));

	fig = plt.figure(figsize = (10 * n_columns, 5 * n_rows));
	gs = fig.add_gridspec(n_rows, n_columns, hspace = 0.45, wspace = 0.3);

	for index, row in dataframe.iterrows():

		row_index = index // n_columns;
		column_index = index % n_columns;
		ax = fig.add_subplot(gs[row_index, column_index]);

		parameter_value = row["folder_name"].split("_")[-1];
		time_range = row["time_range"];
		vpar = row["vpar"];
		delta_f = row["delta_f"];

		vpar_evolution = ax.imshow(
			delta_f.T,
			aspect = "auto",
			origin = "lower",
			extent = [time_range[0], time_range[-1], vpar[0], vpar[-1]],
			cmap = "bwr",
			vmin = -np.max(np.abs(delta_f)),
			vmax =  np.max(np.abs(delta_f)),
		);
		ax.set_title("Velocity-space evolution (Nvpar = 256)");
		ax.set_xlabel("t (GYSELA time-steps)");
		ax.set_ylabel(r"$v_\parallel$ (index space)");
		ax.set_title(f"{parameter_label} = {parameter_value}, $\\rho$ = {effective_radius}");
		ax.legend(frameon = True, loc = "lower right");
		fig.colorbar(vpar_evolution, label = r"$\delta f$", ax = ax);

	# Hide any unused axes in the grid.
	for unused_index in range(n_runs, n_rows * n_columns):
		fig.add_subplot(gs[unused_index // n_columns, unused_index % n_columns]).set_visible(False);

	fig.suptitle(rf"$\delta f$ scaling in velocity-space — parameter scan, {parameter_label}", fontsize = 14);
	plt.tight_layout();
	plt.show();

In [ ]:
dataframe = parameter_scan_delta_f_evolution(BASE_PATH, PREFIX, dt_diag = 50);
plot_parameter_scan_delta_f_evolution(dataframe, n_columns = 3, parameter_label = "nustar");